# Explorative Datenanalyse

## 1. Setup & Datenimport


In [1]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CSV-Datei einlesen (Pfad anpassen)
df = pd.read_csv("data/pv_data.csv", sep=";", parse_dates=["timestamp"])

# Kurzer Überblick
print("Datenvorschau:")
display(df.head())

print("\nAllgemeine Infos:")
df.info()


FileNotFoundError: [Errno 2] No such file or directory: 'data/pv_data.csv'

## 2. Grundlegende Datenprüfung

In [ ]:

print("\nFehlende Werte pro Spalte:")
print(df.isna().sum())

print("\nStatistische Grundwerte:")
display(df.describe())

## 3. Zeitraum & Zeitauflösung

In [ ]:

start = df["timestamp"].min()
ende = df["timestamp"].max()
dauer = ende - start

print(f"\nZeitraum: {start} bis {ende}")
print(f"Dauer: {dauer}")

## 4. Kategorien für Analyse definieren

In [ ]:

# Solarproduktion: alle relevanten Spalten
solar_cols = [
    "Solarproduktion Tracker 1",
    "Solarproduktion Tracker 2",
    "Solarproduktion Tracker 3",
    "Solarproduktion"
]

# Wallbox-Gesamtladeleistung & Solarladeleistung
wallbox_cols = [
    "Wallbox (ID 0) Gesamtladeleistung",
    "Wallbox Gesamtladeleistung",
    "Wallbox (ID 0) Solarladeleistung"
]

## 5. Kennzahlen (Mittel, Min, Max)

In [ ]:

def summary_stats(columns):
    return df[columns].agg(["mean", "min", "max"]).T.rename(
        columns={"mean": "Durchschnitt", "min": "Minimum", "max": "Maximum"}
    )

print("\n--- Solarproduktion ---")
display(summary_stats(solar_cols))

print("\n--- Wallbox-Leistungen ---")
display(summary_stats(wallbox_cols))

## 6. Visualisierung

In [ ]:

sns.set_style("whitegrid")
plt.figure(figsize=(12,6))
plt.plot(df["timestamp"], df["Solarproduktion"], label="Solarproduktion (gesamt)", color="orange")
plt.title("Solarproduktion über Zeit")
plt.xlabel("Zeit")
plt.ylabel("Leistung [kW]")
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df["timestamp"], df["Wallbox Gesamtladeleistung"], label="Wallbox Gesamt", color="blue")
plt.plot(df["timestamp"], df["Wallbox (ID 0) Solarladeleistung"], label="Wallbox Solarladung", color="green")
plt.title("Wallbox-Leistung über Zeit")
plt.xlabel("Zeit")
plt.ylabel("Leistung [kW]")
plt.legend()
plt.show()

## 7. Vergleich: Hausverbrauch (ohne Wallbox) vs. Solarproduktion

In [ ]:


# Neue Spalte: Hausverbrauch ohne Wallbox
df["Hausverbrauch_ohne_Wallbox"] = df["Hausverbrauch"] - df["Wallbox Gesamtladeleistung"]

# Summenvergleich über gesamten Zeitraum
gesamt_solar = df["Solarproduktion"].sum()
gesamt_haus = df["Hausverbrauch_ohne_Wallbox"].sum()

print("\n--- Gesamtvergleich ---")
print(f"Gesamte Solarproduktion: {gesamt_solar:.2f} kWh")
print(f"Gesamter Hausverbrauch ohne Wallbox: {gesamt_haus:.2f} kWh")

deckung = (gesamt_solar / gesamt_haus) * 100 if gesamt_haus > 0 else 0
print(f"Solarproduktion deckt {deckung:.1f}% des Haushaltsverbrauchs (ohne Wallbox)")

## 8. Zeitliche Visualisierung

In [ ]:

plt.figure(figsize=(12,6))
plt.plot(df["timestamp"], df["Solarproduktion"], label="Solarproduktion", color="orange")
plt.plot(df["timestamp"], df["Hausverbrauch"], label="Hausverbrauch", color="red")
plt.title("Vergleich: Solarproduktion vs. Hausverbrauch (ohne Wallbox)")
plt.xlabel("Zeit")
plt.ylabel("Leistung [kW]")
plt.legend()
plt.show()

## 9. Tagesprofile (optional)

In [ ]:

df["hour"] = df["timestamp"].dt.hour
solar_by_hour = df.groupby("hour")["Solarproduktion"].mean()
hausbedarf_by_hour = df.groupby("hour")["Hausverbrauch"].mean()

plt.figure(figsize=(10,5))
plt.plot(solar_by_hour, label="Ø Solarproduktion", color="orange")
plt.plot(hausbedarf_by_hour, label="Ø Hausverbrauch Gesamt", color="blue")
plt.title("Durchschnittlicher Tagesverlauf")
plt.xlabel("Stunde des Tages")
plt.ylabel("Leistung [kW]")
plt.legend()
plt.show()


## 10. Erste Interpretation

In [ ]:

print("\nErste Erkenntnisse:")
print("- Die höchste Solarproduktion liegt typischerweise zwischen 10 und 15 Uhr.")
print("- Optimaler Ladezeitpunkt für E-Fahrzeug ist bei hoher Solarproduktion und geringer Netzbezugslast.")
print("- Nächster Schritt: Prädiktives Modell (z.B. Regression oder Zeitreihenanalyse) für Ladezeit-Vorhersage.")